# Preprocessing

In [3]:
import flopy
import numpy as np

In [4]:
length_units = 'meters'
time_units = 'days'
sim_name = 'ex10_dens'
gwf_name = f'gwf_{sim_name}'
gwt_name = f'gwt_{sim_name}'
gws_name = f'gws_{sim_name}' # salinity for density simulation
sim_ws = './mf6'
concentration_name = 'concentration' # name for concentration
rtmf6_sol_number_name = 'rtmf6_sol_number' # solution number of PHREEQC solution
density_species_name = 'SALINITY' # we add a species SALINITY for the density-dependent simulation

In [5]:
sim = flopy.mf6.MFSimulation(sim_name=sim_name, sim_ws=sim_ws)
# 2 stress periods:
# Period 1 (375 days): Characterized by inflow of a toluene plume that has also a salinity of 35 g/L and corresponding density
# Period 2 (125 days): Stop of contamination, inflow of prisitine groundwater
nper = 2  
perlen = [375,125]
nstp = [7500,2500]

period_data = []
for i in range(nper):
    period_data.append((perlen[i],nstp[i],1.0))

In [6]:
tdis = flopy.mf6.ModflowTdis(
    sim, 
    nper=nper, 
    perioddata=period_data, 
    time_units=time_units)

## Groundwater flow (GWF)

In [7]:
gwf = flopy.mf6.ModflowGwf(
    sim,
    modelname=gwf_name,
    save_flows=True,
    model_nam_file=f"{gwf_name}.nam",
)

In [8]:
#%% Flow solver parameters
nouter, ninner = 600, 900
hclose, rclose, relax = 1e-6, 1e-6, 0.99


imsgwf = flopy.mf6.ModflowIms(
    sim,
    complexity="complex",
    print_option="SUMMARY",
    outer_dvclose=hclose,
    outer_maximum=nouter,
    under_relaxation="NONE",
    inner_maximum=ninner,
    inner_dvclose=hclose,
    rcloserecord=rclose,
    linear_acceleration="CG",
    scaling_method="NONE",
    reordering_method="NONE",
    relaxation_factor=relax,
    filename=f"{gwf_name}.ims",
)
sim.register_ims_package(imsgwf, [gwf.name])

In [9]:
nlay = 100  # Number of layers
ncol = 80 # Number of columns
nrow = 1  # Number of rows
delr = 200.0/ncol # 
delc = 1
top = 50
botm = np.linspace(top-50/nlay,0,nlay)
dis = flopy.mf6.ModflowGwfdis(
    gwf,
    length_units=length_units,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top = top,
    botm = botm,
    filename=f"{gwf_name}.dis",
    nogrb=True,
)

In [10]:
k11 = 10.0  # Horizontal hydraulic conductivity ($m/d$)
k33 = k11  # Vertical hydraulic conductivity ($m/d$)
icelltype = 0 # saturated thickness is constant --> confined case


npf = flopy.mf6.ModflowGwfnpf(
    gwf,
    save_flows=True,
    save_saturation=True,
    icelltype=icelltype,
    k=k11,
    k33=k33,
    save_specific_discharge=True,
    filename=f"{gwf_name}.npf",
)

In [11]:
flopy.mf6.ModflowGwfic(gwf, strt=1, filename=f"{gwf_name}.ic")

package_name = ic
filename = gwf_ex10_dens.ic
package_type = ic
model_or_simulation_package = model
model_name = gwf_ex10_dens

Block griddata
--------------------
strt
{constant 1}



In [12]:
# The water inflow will be via a 2nd type boundary, were a WEL boundary

q = 0.05 #injection rate m3/d
concentration = 0  # will be replaced
rtmf6_sol_number = 0  # use solution from phreeqcrm/advect.pqi

sourcec = np.zeros(nlay)
sourcec[14:30]=1.0 # assign where the toluene plume should enter.

source_salinity = sourcec*35 # here also the salinity should be 35 g/L, elsewhere it is set to zero.


auxiliary = [
    concentration_name, # name for concentration
    rtmf6_sol_number_name, # solution number of PHREEQC solution
    density_species_name
] 


wel_spd_0=[] # stressperiod 1
wel_spd_1=[] # stressperiod 2

for i in range(nlay):
  wel_spd_0.append([(i, 0, 0), q,0,sourcec[i],source_salinity[i]]) # contamination
  wel_spd_1.append([(i, 0, 0), q,0,0,0]) # no contamination
  
wel_data={}
wel_data[0]=wel_spd_0
wel_data[1]=wel_spd_1

wel = flopy.mf6.ModflowGwfwel(
        gwf,
        stress_period_data=wel_data,
        save_flows=True,
        auxiliary=auxiliary,
        pname='wel',
        filename=f"{gwf_name}.wel"
    )


In [13]:
# out flow on the righ side will be via a GHB boundary
headdwn = 51
ghb_spd = []


auxiliary = [
    concentration_name, # name for concentration
    rtmf6_sol_number_name, # solution number of PHREEQC solution
    density_species_name,
    'DENSITY'# for density dependent simulation
]


conductance = 1e6 # to mimic CHD

for i in range(nlay):
  ghb_spd.append([(i, 0, ncol-1), headdwn,conductance,0,0, 0,1000.0])

ghb = flopy.mf6.ModflowGwfghb(
    gwf,
    maxbound=len(ghb_spd),
    stress_period_data=ghb_spd,
    save_flows=False,
    auxiliary=auxiliary,
    pname="GHB",
    filename=f"{gwf_name}.ghb",
)

In [14]:
oc_gwf = flopy.mf6.ModflowGwfoc(
    gwf,
    head_filerecord=f"{gwf_name}.hds",
    budget_filerecord=f"{gwf_name}.cbb",
    headprintrecord=[("COLUMNS", 10, "WIDTH", 15, "DIGITS", 6, "GENERAL")],
    saverecord=[("HEAD", "ALL"), ("BUDGET", "LAST")],
    printrecord=[("HEAD", "LAST"), ("BUDGET", "LAST")],
)

## Transport

In [15]:
# Some parameters:
dispersivity = 0.1
transverse_horizontal_dispersivity = dispersivity * 0.01
transverse_vertical_dispersivity = dispersivity * 0.01

advect='UPSTREAM'

porosity = 0.3

nouter, ninner = 600, 900
hclose, rclose, relax = 1e-6, 1e-6, 0.99

 ### GWT - Reactive transport model

In [16]:
gwt = flopy.mf6.MFModel(
    sim,
    model_type="gwt6",
    modelname=gwt_name,
    model_nam_file=f"{gwt_name}.nam"
)

In [17]:
#%% Transport solver parameters
imsgwt = flopy.mf6.ModflowIms(
    sim,
    print_option="SUMMARY",
    outer_dvclose=hclose,
    outer_maximum=nouter,
    under_relaxation="NONE",
    inner_maximum=ninner,
    inner_dvclose=hclose,
    rcloserecord=rclose,
    linear_acceleration="BICGSTAB",
    scaling_method="NONE",
    reordering_method="NONE",
    relaxation_factor=relax,
    filename=f"{gwt_name}.ims",
)
sim.register_ims_package(imsgwt, [gwt.name])

In [18]:
gwt_dis = flopy.mf6.ModflowGwtdis(
    gwt,
    length_units=length_units,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top = top,
    botm = botm,
    filename=f"{gwt_name}.dis",
    nogrb=True,
)

In [19]:
#%% Transport IC
gwt_ic = flopy.mf6.ModflowGwtic(
    gwt,
    strt=0,  # rtmf6 solution number
    filename=f"{gwt_name}.ic")

In [20]:
#%% Transport SSM
sourcerecarray = ['wel', 'aux', concentration_name]

ssm = flopy.mf6.ModflowGwtssm(
    gwt, 
    sources=sourcerecarray, 
    save_flows=True,
    print_flows=True,
    filename=f"{gwt_name}.ssm"
)

In [21]:
adv = flopy.mf6.ModflowGwtadv(
    gwt,
    scheme=advect,
)

In [22]:
dsp = flopy.mf6.ModflowGwtdsp(
            gwt,
            xt3d_off=True,
            alh=dispersivity,
            ath1=transverse_horizontal_dispersivity,
            atv=transverse_vertical_dispersivity,
            filename=f"{gwt_name}.dsp",
        )

In [23]:
first_order_decay = None

mst = flopy.mf6.ModflowGwtmst(
    gwt,
    porosity=porosity,
    first_order_decay=first_order_decay,
    filename=f"{gwt_name}.mst",
)

In [24]:
oc_gwt = flopy.mf6.ModflowGwtoc(
    gwt,
    budget_filerecord=f"{gwt_name}.cbb",
    concentration_filerecord=f"{gwt_name}.ucn",
    concentrationprintrecord=[
        ("COLUMNS", 10, "WIDTH", 15, "DIGITS", 10, "GENERAL")
    ],
    saverecord=[("CONCENTRATION", "LAST"), #"FREQUENCY",10), 
                ("BUDGET", "LAST")
                ],
    printrecord=[("CONCENTRATION", "LAST"), 
                 ("BUDGET", "LAST")
                    ],
)

In [25]:
#%% GWF-GWT Exchange
flopy.mf6.ModflowGwfgwt(
    sim,
    exgtype="GWF6-GWT6",
    exgmnamea=gwf_name,
    exgmnameb=gwt_name,
    filename=f"{sim_name}.gwfgwt",
)

package_name = ex10_dens.gwfgwt
filename = ex10_dens.gwfgwt
package_type = gwfgwt
model_or_simulation_package = simulation
simulation_name = ex10_dens


### GWS - Density-dependent salinity model

In [26]:
gws = flopy.mf6.MFModel(
    sim,
    model_type="gwt6",
    modelname=gws_name,
    model_nam_file=f"{gws_name}.nam"
)


imsgwt_s = flopy.mf6.ModflowIms(
    sim,
    print_option="SUMMARY",
    outer_dvclose=hclose,
    outer_maximum=nouter,
    under_relaxation="NONE",
    inner_maximum=ninner,
    inner_dvclose=hclose,
    rcloserecord=rclose,
    linear_acceleration="BICGSTAB",
    scaling_method="NONE",
    reordering_method="NONE",
    relaxation_factor=relax,
    filename=f"{gws_name}.ims",
)
sim.register_ims_package(imsgwt_s, [gws.name])

gwt_dis_s = flopy.mf6.ModflowGwtdis(
    gws,
    length_units=length_units,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    delr=delr,
    delc=delc,
    top = top,
    botm = botm,
    filename=f"{gws_name}.dis",
    nogrb=True,
)

gwt_ic_s = flopy.mf6.ModflowGwtic(
    gws, 
    strt=0,  # rtmf6 solution number
    filename=f"{gws_name}.ic")


sourcerecarray = ['wel', 'AUX', density_species_name]
    
ssm_s = flopy.mf6.ModflowGwtssm(
    gws, 
    sources=sourcerecarray, 
    save_flows=True,
    print_flows=True,
    filename=f"{gws_name}.ssm"
)


adv_s = flopy.mf6.ModflowGwtadv(
    gws,
    scheme=advect,
)


dsp_s = flopy.mf6.ModflowGwtdsp(
            gws,
            xt3d_off=True,
            alh=dispersivity,
            ath1=transverse_horizontal_dispersivity,
            atv=transverse_vertical_dispersivity, 
            filename=f"{gws_name}.dsp",
        )

first_order_decay = None

mst_s = flopy.mf6.ModflowGwtmst(
    gws,
    porosity=porosity,
    first_order_decay=first_order_decay,
    filename=f"{gws_name}.mst",
)

oc_gwt_s = flopy.mf6.ModflowGwtoc(
    gws,
    budget_filerecord=f"{gws_name}.cbb",
    concentration_filerecord=f"{gws_name}.ucn",
    concentrationprintrecord=[
        ("COLUMNS", 10, "WIDTH", 15, "DIGITS", 10, "GENERAL")
    ],
    saverecord=[("CONCENTRATION", "LAST"), # "FREQUENCY",10), 
                ("BUDGET", "LAST")
                ],
    printrecord=[("CONCENTRATION", "LAST"), 
                 ("BUDGET", "LAST")
                    ],
)    

gwfgwt_s = flopy.mf6.ModflowGwfgwt(
    sim,
    exgtype="GWF6-GWT6",
    exgmnamea=gwf_name,
    exgmnameb=gws_name,
    filename=f"{sim_name}_s.gwfgwt",
)

In [27]:
#Density Coupling (BUY)
buy = flopy.mf6.ModflowGwfbuy(
  gwf,
  packagedata=[(0, 0.7, 0.0, gws_name, density_species_name)],
)

In [28]:
# writing the MF6 files to be used as templates for the RTMF6 input, such as initial conditions
sim.write_simulation()

writing simulation...
  writing simulation name file...
  writing simulation tdis package...
  writing solution package ims_-1...
  writing solution package ims_0...
  writing solution package ims_1...
  writing package ex10_dens.gwfgwt...
  writing package ex10_dens_s.gwfgwt...
  writing model gwf_ex10_dens...
    writing model name file...
    writing package dis...
    writing package npf...
    writing package ic...
    writing package wel...
INFORMATION: maxbound in ('', 'wel', 'dimensions') changed to 100 based on size of stress_period_data
    writing package ghb...
    writing package oc...
    writing package buy...
INFORMATION: nrhospecies in ('', 'buy', 'dimensions') changed to 1 based on size of packagedata
  writing model gwt_ex10_dens...
    writing model name file...
    writing package dis...
    writing package ic...
    writing package ssm...
    writing package adv...
    writing package dsp...
    writing package mst...
    writing package oc...
  writing model gws_